In [1]:
import csv
import json
import pandas as pd
import numpy as np
from pydantic import ValidationError,field_validator
from datetime import datetime
import requests

In [2]:
import os

os.makedirs("data", exist_ok=True)

In [3]:
def parse_row(row):
    parsed = row.copy()
    Schema = {
        "age":int,
        "fee_inr":int,
        "prior_no_shows":int,
        "distance_km":float
    }
    
    for column,data_type in Schema.items():
        try:
            parsed[column] = data_type(parsed[column])
        except ValueError as e:
            return None,column,e

    return parsed,None,None

    

In [4]:
with open("data/appointment.csv", newline="") as appointment_csv:
    reader = csv.DictReader(appointment_csv)
    headers = reader.fieldnames    
    print(f"Header: {headers}")
    rejected_rows = []
    active_rows = []
    for row in reader:
        result,column,err = parse_row(row)       
        if result is None:            
            #rejected_rows.append({"row":row,"failed_column":column,"Error":err})
            rejected_rows.append(row)
        else:
            active_rows.append(result)

    if headers and "appointment_id" in headers:
        print(f"Total Rows:{reader.line_num-1}")
    else:
        print(f"Total Rows:{reader.line_num}")

    print(len(rejected_rows))


Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total Rows:1020
32


In [5]:
with open("data/rejected_rows.json", "w", encoding="utf-8") as f:
  json.dump(rejected_rows, f, indent=2)

In [6]:
df = pd.read_csv("data/appointment.csv")


In [7]:
df.shape

(1020, 14)

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   str    
 1   patient_id       1020 non-null   str    
 2   clinic_id        1020 non-null   str    
 3   booked_on        1020 non-null   str    
 4   appt_date        1019 non-null   str    
 5   age              1008 non-null   float64
 6   gender           1020 non-null   str    
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   str    
 10  visit_type       1020 non-null   str    
 11  reminder_sent    1020 non-null   str    
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   str    
dtypes: float64(2), int64(2), str(10)
memory usage: 111.7 KB


In [9]:
print("Null counts:\n", df.isnull().sum())

Null counts:
 appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_date           1
age                12
gender              0
distance_km        20
fee_inr             0
booking_channel     0
visit_type          0
reminder_sent       0
prior_no_shows      0
status              4
dtype: int64


In [10]:
print("Duplicate count:", df.duplicated().sum())

Duplicate count: 20


In [11]:
df = df.drop_duplicates()

In [12]:
print("After dropping duplicates:", len(df))

After dropping duplicates: 1000


In [13]:
df["booking_channel"] = df["booking_channel"].str.strip().str.title()

In [14]:
df['appt_date'] = pd.to_datetime(df["appt_date"],errors="coerce")
df['booked_on'] = pd.to_datetime(df["booked_on"],errors="coerce")

In [15]:
df = df.dropna(subset=["appt_date", "status"])
print("After dropping missing dates/status:", len(df))

After dropping missing dates/status: 990


In [16]:
df["age"] = pd.to_numeric(df["age"], errors="coerce")

In [17]:
df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan

In [18]:
median_age = df["age"].median()
print("Calculated Median Age:", median_age)

Calculated Median Age: 48.0


In [19]:
median_distance = df["distance_km"].median()
print(median_distance)

8.25


In [20]:
df["age"] = df["age"].fillna(median_age).round().astype(int)


In [21]:
df["distance_km"] = df["distance_km"].fillna(median_distance)

In [22]:
for col in ["distance_km", "fee_inr"]:
  q1 = df[col].quantile(0.25)
  q3 = df[col].quantile(0.75)

  print(f"Q1 values in {col}: {q1}")
  print(f"Q3 values in {col}: {q3}")

  iqr = q3 - q1
  lower_fence = q1 - 1.5 * iqr
  upper_fence = q3 + 1.5 * iqr

  print(f"Lower fence in {col}: {lower_fence}")
  print(f"Upper fence in {col}: {upper_fence}")

  capped_count = ((df[col] < lower_fence) | (df[col] > upper_fence)).sum()
  print(f"Capped values in {col}: {capped_count}")
  df[col] = np.clip(df[col], lower_fence, upper_fence)
  

Q1 values in distance_km: 4.625
Q3 values in distance_km: 13.0
Lower fence in distance_km: -7.9375
Upper fence in distance_km: 25.5625
Capped values in distance_km: 26
Q1 values in fee_inr: 500.0
Q3 values in fee_inr: 787.5
Lower fence in fee_inr: 68.75
Upper fence in fee_inr: 1218.75
Capped values in fee_inr: 3


In [23]:
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)

In [24]:
df["fee_inr"].max()


np.int64(1219)

In [25]:
df["fee_inr"].min()

np.int64(300)

In [26]:
df["distance_km"].max()

np.float64(25.6)

In [27]:
df["distance_km"].min()

np.float64(0.6)

In [28]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days

In [29]:
df["no_show"] = df["status"].apply(lambda s: 1 if s == "No-Show" else 0)

In [30]:
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

In [31]:
col_means = np.mean(X, axis=0)
print("Column means:", col_means)

Column means: [ 48.17373737   9.50090909 635.71414141  11.81919192]


In [32]:
lead_mask = df["lead_days"].to_numpy() > 14
print(
    f"Lead days > 14: count = {lead_mask.sum()}, share ="
    f" {lead_mask.mean():.4f}"
)

Lead days > 14: count = 296, share = 0.2990


In [33]:
dist_arr = df["distance_km"].to_numpy()
z_scores = (dist_arr - np.mean(dist_arr)) / np.std(dist_arr)
z_count = np.sum(np.abs(z_scores) > 2)
print("Rows with |z| > 2 for distance_km:", z_count)

Rows with |z| > 2 for distance_km: 53


In [34]:
clinics_df = pd.read_csv("data/clinics.csv")

In [35]:
temp_list = []

In [36]:
for _, row in clinics_df.iterrows():
  lat, lon, name = row["latitude"], row["longitude"], row["clinic_name"]
  temp = None
  url = "https://api.open-meteo.com/v1/forecast"
  params = {
      "latitude": lat,
      "longitude": lon,
      "hourly": "temperature_2m",
      "forecast_days": 1,
  }

  for attempt in range(2):
    try:
      res = requests.get(url, params=params, timeout=10)
      if res.status_code == 200:
        data = res.json()
        if "hourly" in data and "temperature_2m" in data["hourly"]:
          temp = data["hourly"]["temperature_2m"][0]
          break
    except requests.exceptions.RequestException as e:
      if attempt == 1:
        print(f"Clinic {name} failed after retry. Reason: {e}")

  temp_list.append(temp)

clinics_df["temp_c"] = temp_list
clinics_df.to_csv("data/clinics_enriched.csv", index=False)

In [37]:
df =df.sort_values("appointment_id")

In [38]:
df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")

In [39]:
appointment_cols = [
    "appointment_id",
    "patient_id", 
    "clinic_id", 
    "booked_on", 
    "appt_date", 
    "age", 
    "gender",
    "distance_km", 
    "fee_inr", 
    "booking_channel", 
    "visit_type", 
    "reminder_sent", 
    "prior_no_shows", 
    "status",
    "lead_days", 
    "no_show"
]

df = df[appointment_cols]

In [40]:
assert df["appointment_id"].nunique() == len(
    df
), "appointment_id is not unique!"
assert df.isnull().sum().sum() == 0, "Missing values found in clean dataset!"

In [41]:
df.to_csv("data/appointments_clean.csv", index=False)

In [42]:
with open("data/load_data.sql", "w", encoding="utf-8") as f:
  
  for _, r in clinics_df.iterrows():
    t_val = "NULL" if pd.isna(r["temp_c"]) else r["temp_c"]
    f.write(
        f"INSERT INTO clinics VALUES ('{r['clinic_id']}','{r['clinic_name']}','"
        f"{r['city']}','{r['tier']}',{r['latitude']},{r['longitude']},{t_val});\n"
    )

  
  for _, r in df.iterrows():
    f.write(
        "INSERT INTO appointments VALUES"
        f" ('{r['appointment_id']}','{r['patient_id']}','{r['clinic_id']}','{r['booked_on']}','{r['appt_date']}',{r['age']},'{r['gender']}',{r['distance_km']},{r['fee_inr']},'{r['booking_channel']}','{r['visit_type']}','{r['reminder_sent']}',{r['prior_no_shows']},'{r['status']}',{r['lead_days']},{r['no_show']});\n"
    )
